# Exercise 3: Hierarchical forecasting of Walmart sales (M5 Forecasting – Accuracy)

**Data:** [M5 Forecasting – Accuracy (Kaggle)](https://www.kaggle.com/competitions/m5-forecasting-accuracy): daily unit sales of 3,049 Walmart products in 10 stores, with calendar/event, SNAP and price data.

This notebook answers the four questions of the exercise. Each answer is written up in markdown and backed by code that runs on the real data:

| Question | Section |
|---|---|
| 1. Full approach to building the forecasting model | [Q1](#q1) |
| 2. Key dimensions and hierarchy levels and how they shape modelling choices | [Q2](#q2) |
| 3. Exploratory visualisations: trend, seasonality, calendar/holiday effects and intermittency | [Q3](#q3) |
| 4. Models, transformations and exogenous variables, with reasons | [Q4](#q4) |

It builds on my earlier notebook `lgbm-model.ipynb` (LightGBM + Optuna) and on Google's *Vertex AI Training for LightGBM* sample. The companion notebook `m5_vertex_custom_training.ipynb` runs the same pipeline as a custom training job on Vertex AI (Gemini Enterprise Agent Platform).

**How to run.** On Kaggle, attach the competition data (default path below). Elsewhere (Colab, Vertex AI Workbench, local), the notebook downloads it with `kagglehub` if you have a Kaggle API token, or you can set `M5_DATA_PATH` to a local folder or a `gs://` bucket. I estimate a full run on the real data at 1–2 h on a 4-core / 30 GB Kaggle machine, with peak memory around 10 GB (from scaling tests). Set `M5_FAST_MODE=1` for a quicker pass with a shorter training window and fewer boosting rounds.

<a id="q1"></a>
## Q1. Full approach to building a forecasting model for M5

To build a forecasting model for this M5-Forecasting dataset I would first start with a methodological approach to managing hierarchical time series data. The task is to forecast **28 days of daily unit sales for 30,490 item × store series**. Forecasts are scored with the **WRMSSE** (weighted root mean squared scaled error) on **all 42,840 series** of a 12-level hierarchy, so a good forecast must be accurate *and* add up at every level, from a single product in one store to total Walmart sales. My approach has eight steps:

| # | Step | What I do | Why it matters for M5 |
|---|---|---|---|
| 1 | **Frame the problem and metric** | Reproduce WRMSSE in code (weights = last-28-day dollar sales, scale = in-sample naive error) and report it per level. | Every later choice (features, loss, reconciliation) is judged by the score the competition uses, level by level. |
| 2 | **Prepare the data** | Load sales into an `(series × days)` matrix. Join the calendar (events, SNAP) and weekly prices. Treat days before an item's launch (no price) as *not on sale*, not as zero demand. Flag Christmas store closures. | Avoids training on structural zeros. Keeps memory small: the long format is ~59M rows. |
| 3 | **Explore by hierarchy level** (Q2, Q3) | Trend, weekly and annual seasonality, events, SNAP, prices and intermittency at several levels. | The findings decide the feature set, the loss function and which model goes at which level. |
| 4 | **Design validation like the real forecast** | Rolling-origin backtest with 28-day windows: fold A (d_1886–1913) for early stopping and tuning, fold B (d_1914–1941) as an untouched holdout. Features may only use data available at the forecast origin (sales lags ≥ 28 days). | Random splits leak the future. Validating with information you won't have at forecast time gives optimistic scores (a problem in my first notebook). |
| 5 | **Benchmarks** | Seasonal naive (repeat last week) for every series. ETS at the aggregate levels. | Any complex model has to beat these, level by level. |
| 6 | **Main model** | One **global LightGBM** trained on all 30,490 item-store series with a **Tweedie** loss. Features: lagged and rolling demand, calendar, events, state-specific SNAP, price/promotion and hierarchy-level demand. | Bottom series are short, noisy and intermittent. A global model shares strength across items and stores and handles zeros with a count-aware loss. |
| 7 | **Make forecasts coherent across the hierarchy** | Compare bottom-up LightGBM with ETS/ARIMA forecasts at levels 1–9 that are **reconciled** (WLS) and pushed down to items **middle-out**. Choose by holdout WRMSSE. | Top levels are smooth and well served by statistical models. Item level needs the ML model. Reconciliation combines both into one coherent set. |
| 8 | **Refit, forecast, productionise** | Refit through d_1941, forecast d_1942–1969, write the submission. Run the same code as a Vertex AI custom training job (companion notebook), scheduled each forecast cycle, and monitor WRMSSE per level. | Reproducible and automated. It also fixes the train/serve differences of my first notebook. |

**Next improvements once the baseline pipeline is in place:** Optuna tuning on the backtest folds with WRMSSE as the objective; per-store or per-horizon models; seed and model ensembles; probabilistic forecasts (quantiles) for the M5 *Uncertainty* track.

**Why LightGBM rather than the platform's AutoML forecasting?** AutoML hierarchical forecasting on Vertex AI is a useful managed benchmark. LightGBM through custom training gives full control over the features, the loss, the validation scheme and reconciliation. Gradient-boosted trees with this kind of feature set were the backbone of most top M5 solutions.

## Setup

In [ ]:
import os, gc, time, warnings, importlib
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, PercentFormatter
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

In [ ]:
# ---- Configuration ---------------------------------------------------------------
DATA_PATH = os.environ.get("M5_DATA_PATH", "/kaggle/input/m5-forecasting-accuracy")
if not DATA_PATH.startswith("gs://") and not os.path.exists(DATA_PATH):
    # Outside Kaggle: download the competition files with kagglehub (needs a Kaggle API token
    # and the competition rules accepted on kaggle.com)
    import kagglehub
    DATA_PATH = kagglehub.competition_download("m5-forecasting-accuracy")
OUTPUT_DIR = os.environ.get("M5_OUTPUT_DIR", ".")
FIG_DIR = os.path.join(OUTPUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

FAST_MODE = os.environ.get("M5_FAST_MODE", "0") == "1"
# training window (days) for each LightGBM fit, upper bound on boosting rounds (early stopping
# picks the actual number) and learning rate; each can also be set through an environment variable
LOOKBACK_DAYS = int(os.environ.get("M5_LOOKBACK_DAYS", 364 if FAST_MODE else 728))
MAX_ROUNDS = int(os.environ.get("M5_MAX_ROUNDS", 400 if FAST_MODE else 1500))
LEARNING_RATE = float(os.environ.get("M5_LEARNING_RATE", 0.08 if FAST_MODE else 0.05))
LGB_OVERRIDES = {"learning_rate": LEARNING_RATE}
print(f"DATA_PATH={DATA_PATH}  LOOKBACK_DAYS={LOOKBACK_DAYS}  MAX_ROUNDS={MAX_ROUNDS}  LEARNING_RATE={LEARNING_RATE}")

In [ ]:
# ---- Chart style -------------------------------------------------------------------
# Categorical slots 1-4 (validated for colour-vision deficiency, in this order), a light blue
# for raw daily values under a rolling mean, and recessive ink/grid colours.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
BLUE_LIGHT, POS, NEG = "#86b6ef", "#2a78d6", "#e34948"
INK, INK2, MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.linewidth": 0.8, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.titlecolor": INK,
    "axes.labelsize": 9, "axes.labelcolor": INK2, "xtick.color": MUTED, "ytick.color": MUTED,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "text.color": INK, "legend.frameon": False,
    "legend.fontsize": 8, "lines.linewidth": 1.6, "figure.dpi": 110, "font.family": "sans-serif",
    "axes.prop_cycle": mpl.cycler(color=SERIES),
})

def save(fig, name):
    fig.savefig(os.path.join(FIG_DIR, f"{name}.png"), dpi=150, bbox_inches="tight")

def _k(v, _=None):
    if abs(v) < 1000:
        return f"{v:,.0f}"
    return f"{v / 1000:,.0f}k" if v % 1000 == 0 or abs(v) >= 10000 else f"{v / 1000:,.1f}k"

def thousands(ax, axis="y"):
    (ax.yaxis if axis == "y" else ax.xaxis).set_major_formatter(FuncFormatter(_k))

def end_label(ax, x, y, text):
    ax.annotate(text, (x, y), xytext=(4, 0), textcoords="offset points", va="center", fontsize=8, color=INK2)

def short_id(series_id):
    return series_id.replace("_evaluation", "")

### Core pipeline module

The data loading, hierarchy, WRMSSE metric, feature builder, models and reconciliation live in one module. This notebook and the Vertex AI training job (`m5_vertex_custom_training.ipynb`) then run exactly the same code. The cell writes `m5_core.py` next to the notebook and imports it.

In [ ]:
%%writefile m5_core.py
"""Core M5 forecasting pipeline shared by the Exercise 3 notebook and the Vertex AI trainer.

Generated by exercise3_m5_forecasting.ipynb (the `%%writefile m5_core.py` cell).
Edit the notebook cell and re-run it rather than editing this file by hand.

Contents
- load_m5 / price_matrix          : read the three Kaggle files into compact arrays
- build_summing_matrix / LEVELS   : the 12-level M5 hierarchy as a sparse summing matrix S
- WRMSSE                          : the official accuracy metric, evaluated on all 42,840 series
- FeatureBuilder                  : horizon-safe features (every sales feature is lagged >= 28 days)
- train_lgbm / predict_days       : global LightGBM (Tweedie) model
- seasonal_naive / ets_forecast   : benchmarks and upper-level statistical models
- fill_closed_days                : impute Christmas store closures for statistical models
- hierarchical_variants           : bottom-up vs. WLS-reconciled + middle-out forecasts
- make_submission                 : Kaggle submission frame
"""
import warnings

import numpy as np
import pandas as pd
from scipy import sparse

HORIZON = 28
ID_COLS = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]

# The 12 aggregation levels used by the M5 WRMSSE metric: level -> (name, grouping columns).
LEVELS = {
    1: ("Total", []),
    2: ("State", ["state_id"]),
    3: ("Store", ["store_id"]),
    4: ("Category", ["cat_id"]),
    5: ("Department", ["dept_id"]),
    6: ("State x Category", ["state_id", "cat_id"]),
    7: ("State x Department", ["state_id", "dept_id"]),
    8: ("Store x Category", ["store_id", "cat_id"]),
    9: ("Store x Department", ["store_id", "dept_id"]),
    10: ("Item", ["item_id"]),
    11: ("Item x State", ["item_id", "state_id"]),
    12: ("Item x Store", ["item_id", "store_id"]),
}


# --------------------------------------------------------------------------------------
# Data loading
# --------------------------------------------------------------------------------------
def load_m5(data_path):
    """Read the Kaggle files.

    Returns
    -------
    meta : DataFrame (n_series rows) with the id/hierarchy columns, in file order
    Y : float32 array (n_series, n_days) of unit sales, d_1 .. d_n
    calendar : DataFrame with an extra integer `d_num` column
    prices : DataFrame (store_id, item_id, wm_yr_wk, sell_price)
    """
    header = pd.read_csv(f"{data_path}/sales_train_evaluation.csv", nrows=0).columns
    day_cols = [c for c in header if c.startswith("d_")]
    sales = pd.read_csv(
        f"{data_path}/sales_train_evaluation.csv",
        dtype={**{c: np.int16 for c in day_cols}, **{c: str for c in ID_COLS if c in header}},
    )
    meta = sales[[c for c in ID_COLS if c in sales.columns]].reset_index(drop=True)
    if "id" not in meta.columns:  # some public mirrors of the data drop the id column
        meta.insert(0, "id", meta["item_id"] + "_" + meta["store_id"] + "_evaluation")
    Y = sales[day_cols].to_numpy(dtype=np.float32)
    del sales

    calendar = pd.read_csv(f"{data_path}/calendar.csv", parse_dates=["date"])
    if "d" not in calendar.columns:  # ... or the day label; d_1 is the first date
        calendar = calendar.sort_values("date").reset_index(drop=True)
        calendar["d"] = "d_" + (calendar.index + 1).astype(str)
    calendar["d_num"] = calendar["d"].str[2:].astype(int)
    calendar = calendar.sort_values("d_num").reset_index(drop=True)

    prices = pd.read_csv(
        f"{data_path}/sell_prices.csv",
        dtype={"store_id": str, "item_id": str, "wm_yr_wk": np.int32, "sell_price": np.float32},
    )
    return meta, Y, calendar, prices


def price_matrix(meta, calendar, prices):
    """Weekly sell-price matrix (n_series, n_weeks) and the week index of every calendar day.

    A missing price means the item was not on sale in that store that week (e.g. before launch).
    """
    weeks = np.sort(calendar["wm_yr_wk"].unique())
    rows = pd.MultiIndex.from_frame(meta[["item_id", "store_id"]]).get_indexer(
        pd.MultiIndex.from_frame(prices[["item_id", "store_id"]])
    )
    cols = np.searchsorted(weeks, prices["wm_yr_wk"].to_numpy())
    keep = rows >= 0
    P_week = np.full((len(meta), len(weeks)), np.nan, np.float32)
    P_week[rows[keep], cols[keep]] = prices["sell_price"].to_numpy(np.float32)[keep]
    day_week = np.searchsorted(weeks, calendar["wm_yr_wk"].to_numpy())
    return P_week, day_week


# --------------------------------------------------------------------------------------
# Hierarchy and metric
# --------------------------------------------------------------------------------------
def build_summing_matrix(meta, levels=LEVELS):
    """Sparse 0/1 summing matrix S (n_all_series, n_bottom) and a frame describing each row.

    Row order: level 1 first, level 12 last. Level-12 rows are in the same order as `meta`,
    so S @ Y aggregates the bottom-level matrix to every node of the hierarchy at once.
    """
    blocks, info = [], []
    for level, (name, cols) in levels.items():
        if cols:
            key = meta[cols[0]].astype(str)
            for c in cols[1:]:
                key = key + "_" + meta[c].astype(str)
        else:
            key = pd.Series("Total", index=meta.index)
        codes, uniques = pd.factorize(key)
        blocks.append(
            sparse.csr_matrix(
                (np.ones(len(meta), np.float32), (codes, np.arange(len(meta)))),
                shape=(len(uniques), len(meta)),
            )
        )
        info.append(pd.DataFrame({"level": level, "level_name": name, "series": np.asarray(uniques)}))
    return sparse.vstack(blocks).tocsr(), pd.concat(info, ignore_index=True)


class WRMSSE:
    """M5 Weighted Root Mean Squared Scaled Error for the 28 days after `train_end`.

    - scale: mean squared one-step naive error of each series, from its first non-zero sale
      up to `train_end` (so series are comparable regardless of volume)
    - weight: dollar sales over the last 28 training days, normalised within each level,
      and every level counts 1/12 in the total
    """

    def __init__(self, Y, P_day, S, info, train_end, chunk=4000):
        self.S, self.info, self.train_end = S, info, train_end
        hist = Y[:, :train_end]
        self.actual = np.asarray(S @ Y[:, train_end:train_end + HORIZON], dtype=np.float64)

        scale = np.empty(S.shape[0])
        for start in range(0, S.shape[0], chunk):
            A = np.asarray(S[start:start + chunk] @ hist, dtype=np.float64)
            started = np.cumsum(A != 0, axis=1)[:, :-1] > 0
            d2 = np.diff(A, axis=1) ** 2
            scale[start:start + chunk] = (d2 * started).sum(1) / np.maximum(started.sum(1), 1)
        self.scale = np.where(scale > 0, scale, np.nan)

        dollars = np.nansum(hist[:, -HORIZON:] * P_day[:, train_end - HORIZON:train_end], axis=1)
        agg = np.asarray(S @ dollars, dtype=np.float64)
        level_total = pd.Series(agg).groupby(info["level"].to_numpy()).transform("sum").to_numpy()
        self.weights = agg / level_total

    def rmsse(self, pred_bottom):
        agg = np.asarray(self.S @ pred_bottom, dtype=np.float64)
        return np.sqrt(np.mean((self.actual - agg) ** 2, axis=1) / self.scale)

    def score(self, pred_bottom):
        """Return (overall WRMSSE, per-level WRMSSE Series indexed by level)."""
        contrib = np.nan_to_num(self.weights * self.rmsse(pred_bottom))
        per_level = pd.Series(contrib).groupby(self.info["level"].to_numpy()).sum()
        return float(per_level.mean()), per_level


# --------------------------------------------------------------------------------------
# Features
# --------------------------------------------------------------------------------------
LAGS = [28, 35, 42, 49, 56]
WINDOWS = [7, 14, 28, 56, 112]
STATIC_CATS = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]
EVENT_CATS = ["event_name_1", "event_type_1", "event_name_2"]


def _cumsum0(A, dtype=np.float64):
    """Cumulative sum along days with a leading zero column: sum(A[:, a:b]) = C[:, b] - C[:, a]."""
    C = np.zeros((A.shape[0], A.shape[1] + 1), dtype)
    np.cumsum(A, axis=1, dtype=dtype, out=C[:, 1:])
    return C


def _window_sum(C, end, w):
    """Sum over the w days ending at 0-based day `end` (array); NaN where the window starts before day 0."""
    start = end - w + 1
    out = C[:, end + 1] - C[:, np.clip(start, 0, None)]
    out[:, start < 0] = np.nan
    return out


class FeatureBuilder:
    """Builds the LightGBM design matrix for any set of target days.

    Horizon-safe by construction: every feature derived from sales uses only days at least
    HORIZON (28) days before the target day. One model therefore forecasts all 28 days
    directly (no recursion, no error feedback), and a backtest sees exactly the information
    that is available when the real forecast is made. Calendar and price features are known
    in advance in M5 (calendar.csv and sell_prices.csv cover the forecast period).

    Rows are series-major (all target days of series 0, then series 1, ...) in `meta` order,
    so predictions reshape straight back to an (n_series, n_days) matrix.
    """

    def __init__(self, meta, Y, calendar, P_week, day_week):
        self.n, self.n_days = Y.shape
        self.Y = Y
        self.C = _cumsum0(Y)
        self.C2 = _cumsum0(np.square(Y, dtype=np.float64))
        self.Cz = _cumsum0((Y == 0).astype(np.float32), np.float32)

        self.codes = {c: pd.factorize(meta[c])[0].astype(np.float32) for c in STATIC_CATS}
        # Hierarchy-informed demand signals: the same item across all stores, and the
        # store x department it belongs to (average per series).
        self.group_of, self.group_C = {}, {}
        for name, cols in [("item", ["item_id"]), ("store_dept", ["store_id", "dept_id"])]:
            g = pd.factorize(meta[cols].astype(str).agg("_".join, axis=1))[0]
            G = sparse.csr_matrix((np.ones(self.n, np.float32), (g, np.arange(self.n))))
            mean = (G @ Y) / np.asarray(G.sum(axis=1))
            self.group_of[name], self.group_C[name] = g, _cumsum0(mean)

        cal = calendar.sort_values("d_num").reset_index(drop=True)
        self.calendar = cal
        self.wday = cal["wday"].to_numpy(np.float32)
        self.month = cal["month"].to_numpy(np.float32)
        self.mday = cal["date"].dt.day.to_numpy(np.float32)
        self.week = cal["date"].dt.isocalendar().week.to_numpy(np.float32)
        self.events = {}
        for c in EVENT_CATS:
            codes = pd.Categorical(cal[c]).codes.astype(np.float32)
            codes[codes < 0] = np.nan
            self.events[c] = codes
        # Demand moves the day before/after big holidays (e.g. the eve of Thanksgiving), so the
        # model also sees the next and previous day's event; the calendar is known in advance.
        ev = self.events["event_name_1"]
        self.event_next = np.append(ev[1:], np.nan).astype(np.float32)
        self.event_prev = np.insert(ev[:-1], 0, np.nan).astype(np.float32)
        states = pd.unique(meta["state_id"])
        snap = np.stack([cal[f"snap_{s}"].to_numpy(np.float32) for s in states])
        self.snap = snap[pd.Categorical(meta["state_id"], categories=states).codes]

        # Prices: daily matrix plus item-store level summaries.
        self.P_day = P_week[:, day_week]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            self.p_max = np.nanmax(P_week, axis=1, keepdims=True)
            self.p_mean = np.nanmean(P_week, axis=1, keepdims=True)
        prev = np.concatenate([np.full((self.n, 1), np.nan, np.float32), P_week[:, :-1]], axis=1)
        self.p_momentum = (P_week / prev)[:, day_week]
        self.p_nunique = pd.DataFrame(P_week).nunique(axis=1).to_numpy(np.float32)[:, None]
        on_sale = ~np.isnan(P_week)
        first_week = np.where(on_sale.any(1), on_sale.argmax(1), P_week.shape[1]).astype(np.float32)
        self.weeks_since_release = day_week[None, :].astype(np.float32) - first_week[:, None]

        self.feature_names = [name for name, _ in self._columns(np.array([HORIZON]))]
        self.categorical_features = STATIC_CATS + EVENT_CATS + ["event_next", "event_prev"]

    def on_sale(self, days):
        """Boolean (n_series, len(days)) mask: item had a sell price that day."""
        return ~np.isnan(self.P_day[:, np.asarray(days) - 1])

    def _columns(self, t):
        """Yield (name, array) pairs; arrays are (n, T), (n, 1) or (T,) and broadcast to (n, T)."""
        for c in STATIC_CATS:
            yield c, self.codes[c][:, None]
        yield "wday", self.wday[t]
        yield "month", self.month[t]
        yield "mday", self.mday[t]
        yield "week", self.week[t]
        for c in EVENT_CATS:
            yield c, self.events[c][t]
        yield "event_next", self.event_next[t]
        yield "event_prev", self.event_prev[t]
        yield "snap", self.snap[:, t]

        price = self.P_day[:, t]
        yield "sell_price", price
        yield "price_rel_max", price / self.p_max
        yield "price_rel_mean", price / self.p_mean
        yield "price_momentum", self.p_momentum[:, t]
        yield "price_nunique", self.p_nunique
        yield "weeks_since_release", self.weeks_since_release[:, t]

        end = t - HORIZON  # last day a feature may look at (0-based)
        for lag in LAGS:
            src = t - lag
            col = self.Y[:, np.clip(src, 0, None)].copy()
            col[:, src < 0] = np.nan
            yield f"lag_{lag}", col
        for w in WINDOWS:
            yield f"rmean_{w}", _window_sum(self.C, end, w) / w
        s1, s2 = _window_sum(self.C, end, 28), _window_sum(self.C2, end, 28)
        yield "rstd_28", np.sqrt(np.clip((s2 - s1 ** 2 / 28) / 27, 0, None))
        yield "zero_frac_28", _window_sum(self.Cz, end, 28) / 28
        for name in ["item", "store_dept"]:
            group_mean = _window_sum(self.group_C[name], end, 28) / 28
            yield f"{name}_rmean_28", group_mean[self.group_of[name]]

    def make(self, days, require_target=True):
        """Design matrix for 1-based target `days`.

        require_target=True  -> training rows only: item on sale and the target is observed
        require_target=False -> every (series, day) row, for prediction
        Returns X (float32, rows x features), y (float32 or None), row mask (n, T).
        """
        t = np.asarray(days) - 1
        if (t - HORIZON).max() >= self.n_days:
            raise ValueError("target day too far ahead: features need sales up to 28 days before")
        T = len(t)
        if require_target:
            if t.max() >= self.n_days:
                raise ValueError("training days must have observed sales")
            mask = self.on_sale(days)
        else:
            mask = np.ones((self.n, T), bool)
        flat = mask.ravel()
        X = np.empty((int(flat.sum()), len(self.feature_names)), np.float32)
        for j, (_, arr) in enumerate(self._columns(t)):
            X[:, j] = np.broadcast_to(arr, (self.n, T)).ravel()[flat]
        y = self.Y[:, t].ravel()[flat] if require_target else None
        return X, y, mask


# --------------------------------------------------------------------------------------
# Models
# --------------------------------------------------------------------------------------
LGB_PARAMS = {
    "objective": "tweedie",  # zero-inflated, right-skewed counts; predicts the mean
    "tweedie_variance_power": 1.1,
    "metric": "rmse",
    "learning_rate": 0.05,
    "num_leaves": 255,
    "min_data_in_leaf": 100,
    "feature_fraction": 0.7,
    "bagging_fraction": 0.7,
    "bagging_freq": 1,
    "lambda_l2": 0.1,
    "max_bin": 255,
    "verbose": -1,
    "seed": 42,
}


def train_lgbm(fb, train_days, valid_days=None, params=None, num_boost_round=1500,
               early_stopping_rounds=100, log_period=100):
    """Fit the global LightGBM model on `train_days` (optionally early-stopping on `valid_days`)."""
    import lightgbm as lgb

    params = {**LGB_PARAMS, **(params or {})}
    X, y, _ = fb.make(train_days)
    dtrain = lgb.Dataset(X, y, feature_name=fb.feature_names,
                         categorical_feature=fb.categorical_features, free_raw_data=True)
    valid_sets, callbacks = [], [lgb.log_evaluation(log_period)]
    if valid_days is not None:
        Xv, yv, _ = fb.make(valid_days)
        valid_sets = [lgb.Dataset(Xv, yv, reference=dtrain)]
        callbacks.append(lgb.early_stopping(early_stopping_rounds, verbose=True))
    model = lgb.train(params, dtrain, num_boost_round=num_boost_round,
                      valid_sets=valid_sets, callbacks=callbacks)
    del X, y, dtrain
    return model


def predict_days(model, fb, days):
    """Forecast matrix (n_series, len(days)); zero where the item is not on sale."""
    X, _, _ = fb.make(days, require_target=False)
    pred = model.predict(X).reshape(fb.n, len(days))
    pred[~fb.on_sale(days)] = 0.0
    return np.clip(pred, 0, None).astype(np.float32)


def seasonal_naive(Y, train_end, horizon=HORIZON, season=7):
    """Repeat the last observed week."""
    return np.tile(Y[:, train_end - season:train_end], horizon // season).astype(np.float32)


def christmas_mask(calendar, n_days):
    """Boolean mask over d_1..d_n_days marking Christmas Day (stores closed)."""
    cal = calendar.sort_values("d_num").head(n_days)
    return ((cal["date"].dt.month == 12) & (cal["date"].dt.day == 25)).to_numpy()


def fill_closed_days(A, closed):
    """Replace days when stores were closed (Christmas) in each row of A (n, T) by the mean of
    the same weekday one week before and after, so a one-day closure does not distort the
    level, trend or seasonal estimates of statistical models. `closed` is a mask over the T days."""
    A = np.array(A, dtype=np.float64, copy=True)
    T = A.shape[1]
    for k in np.where(closed)[0]:
        nb = [j for j in (k - 7, k + 7) if 0 <= j < T]
        A[:, k] = A[:, nb].mean(axis=1)
    return A


def ets_forecast(series, horizon=HORIZON, closed=None, fit_days=728):
    """Damped-trend additive Holt-Winters with a weekly season, fitted to each row of `series`
    (n, T) over the last `fit_days` days. `closed` (mask over the T days) marks store closures."""
    from statsmodels.tsa.holtwinters import ExponentialSmoothing

    series = np.asarray(series, dtype=np.float64)[:, -fit_days:]
    if closed is not None:
        series = fill_closed_days(series, closed[-series.shape[1]:])
    out = np.zeros((series.shape[0], horizon))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for i, y in enumerate(series):
            fit = ExponentialSmoothing(y, trend="add", damped_trend=True, seasonal="add",
                                       seasonal_periods=7, initialization_method="estimated").fit()
            out[i] = fit.forecast(horizon)
    return np.clip(out, 0, None)


# --------------------------------------------------------------------------------------
# Hierarchical reconciliation
# --------------------------------------------------------------------------------------
def upper_hierarchy(S, info, bottom_level=9):
    """Levels 1..bottom_level as their own hierarchy with Store x Dept nodes as the bottom.

    Returns (rows of S for levels <= bottom_level, S_up (n_upper, n_mid) 0/1 matrix,
    mid-level group index of every item-store series).
    """
    levels = info["level"].to_numpy()
    up_rows = np.where(levels <= bottom_level)[0]
    S_mid = S[np.where(levels == bottom_level)[0]]
    group_of = S_mid.tocsc().indices  # exactly one mid-level parent per item-store
    S_up = ((S[up_rows] @ S_mid.T).toarray() > 0).astype(np.float64)
    return up_rows, S_up, group_of


def reconcile_wls_struct(base, S_up):
    """WLS reconciliation with structural scaling: W = diag(number of bottom series per node).

    Returns coherent bottom-level (mid-level) forecasts G @ base, G = (S'W^-1 S)^-1 S'W^-1.
    """
    StW = S_up.T / S_up.sum(axis=1)
    G = np.linalg.solve(StW @ S_up, StW)
    return G @ base


def middle_out(pred_bottom, mid_forecast, group_of):
    """Split mid-level forecasts to item-stores in proportion to the bottom-level forecasts."""
    denom = np.zeros_like(mid_forecast, dtype=np.float64)
    np.add.at(denom, group_of, pred_bottom)
    ratio = np.divide(mid_forecast, denom, out=np.zeros_like(denom), where=denom > 0)
    return (pred_bottom * ratio[group_of]).astype(np.float32)


def hierarchical_variants(lgbm_bottom, Y, S, info, train_end, closed, ets_fit_days=728):
    """Bottom-level forecasts (n, 28) for each hierarchical strategy, plus the ETS upper forecasts."""
    up_rows, S_up, group_of = upper_hierarchy(S, info)
    S_upper = S[up_rows]
    ets_up = ets_forecast(S_upper @ Y[:, :train_end], closed=closed[:train_end], fit_days=ets_fit_days)
    lgb_up = np.asarray(S_upper @ lgbm_bottom, dtype=np.float64)
    variants = {"LightGBM bottom-up": lgbm_bottom}
    for name, base in [("ETS reconciled + middle-out", ets_up),
                       ("LightGBM+ETS reconciled + middle-out", 0.5 * (ets_up + lgb_up))]:
        mid = np.clip(reconcile_wls_struct(base, S_up), 0, None)
        variants[name] = middle_out(lgbm_bottom, mid, group_of)
    return variants, ets_up


# --------------------------------------------------------------------------------------
# Submission
# --------------------------------------------------------------------------------------
def make_submission(meta, Y, forecast):
    """Kaggle format: `_validation` rows hold the known d_1914..d_1941 actuals,
    `_evaluation` rows hold the 28-day forecast."""
    cols = [f"F{i}" for i in range(1, HORIZON + 1)]
    ids = meta["id"].astype(str)
    evaluation = pd.DataFrame(forecast, columns=cols)
    evaluation.insert(0, "id", ids.to_numpy())
    validation = pd.DataFrame(Y[:, -HORIZON:], columns=cols)
    validation.insert(0, "id", ids.str.replace("_evaluation", "_validation").to_numpy())
    return pd.concat([validation, evaluation], ignore_index=True)

In [ ]:
import m5_core as core
importlib.reload(core);

### Load the data

In [ ]:
t0 = time.time()
meta, Y, calendar, prices = core.load_m5(DATA_PATH)
P_week, day_week = core.price_matrix(meta, calendar, prices)
P_day = P_week[:, day_week]                      # daily price per item-store (NaN = not on sale)
del prices; gc.collect()

n_series, n_days = Y.shape
dates = pd.DatetimeIndex(calendar["date"].iloc[:n_days])
cal_n = calendar.iloc[:n_days].reset_index(drop=True)
closed = core.christmas_mask(calendar, n_days)  # Christmas Day: stores closed
S, info = core.build_summing_matrix(meta)        # 42,840 x 30,490 summing matrix

print(f"{n_series:,} item-store series x {n_days:,} days ({dates[0].date()} to {dates[-1].date()}), "
      f"loaded in {time.time() - t0:.0f}s")
print(f"Sales matrix: {Y.nbytes / 1e9:.2f} GB as float32")

<a id="q2"></a>
## Q2. Key dimensions, hierarchy levels and their effect on modelling choices

**Dimensions**

- **Product:** 3 categories (FOODS, HOBBIES, HOUSEHOLD) → 7 departments → 3,049 items.
- **Location:** 3 states (CA, TX, WI) → 10 stores (4 in CA, 3 in TX, 3 in WI).
- **Time:** daily sales for 1,941 days (2011-01-29 to 2016-05-22), forecast horizon 28 days.
- **Drivers:** a calendar with sporting, cultural, national and religious events and state-specific SNAP (food-stamp) days, plus weekly sell prices per item and store.

Product and location are **crossed**, not nested: the same item is sold in every store. M5 is therefore a *grouped* hierarchy. The WRMSSE metric uses 12 aggregation levels:

In [ ]:
level_table = info.groupby(["level", "level_name"]).size().rename("series").reset_index()
level_table["example series"] = info.groupby("level")["series"].first().to_numpy()
level_table = pd.concat([level_table, pd.DataFrame(
    [{"level": "", "level_name": "All levels", "series": level_table["series"].sum(), "example series": ""}])],
    ignore_index=True)
display(level_table)

print(f"items={meta.item_id.nunique():,}  departments={meta.dept_id.nunique()}  categories={meta.cat_id.nunique()}  "
      f"stores={meta.store_id.nunique()}  states={meta.state_id.nunique()}")

In [ ]:
# Where the volume and the metric weight sit
year = slice(n_days - 364, n_days)
units = pd.DataFrame({"store_id": meta.store_id, "cat_id": meta.cat_id, "units": Y[:, year].sum(1) / 364})
by_store = units.pivot_table(index="store_id", columns="cat_id", values="units", aggfunc="sum", observed=True)
cats = by_store.sum().sort_values(ascending=False).index.tolist()

dollars = np.nansum(Y[:, -28:] * P_day[:, n_days - 28:n_days], axis=1)   # level-12 WRMSSE weights
w = np.sort(dollars)[::-1] / dollars.sum()
cum_w = np.cumsum(w)
share_series = np.arange(1, n_series + 1) / n_series

fig, axes = plt.subplots(1, 2, figsize=(13, 4.3), gridspec_kw={"width_ratios": [1.25, 1]})
ax = axes[0]
left = np.zeros(len(by_store))
for i, c in enumerate(cats):
    ax.barh(by_store.index, by_store[c], left=left, color=SERIES[i], height=0.7,
            edgecolor=SURFACE, linewidth=1.5, label=c)
    left += by_store[c].to_numpy()
ax.invert_yaxis(); ax.grid(axis="x"); ax.grid(axis="y", visible=False); thousands(ax, "x")
ax.set_title("Average daily units by store and category (last 52 weeks)")
ax.set_xlabel("units per day")
ax.legend(loc="lower right", ncol=3)

ax = axes[1]
ax.plot(share_series * 100, cum_w * 100, color=SERIES[0], label="Item-store series")
ax.plot([0, 100], [0, 100], color=AXIS, lw=1, label="Equal weights")
k10 = int(0.10 * n_series) - 1
ax.scatter([10], [cum_w[k10] * 100], color=SERIES[0], s=36, zorder=3, edgecolor=SURFACE, linewidth=1.5)
ax.annotate(f"top 10% of series = {cum_w[k10]:.0%} of weight", (10, cum_w[k10] * 100), xytext=(8, -14),
            textcoords="offset points", fontsize=8, color=INK2)
ax.xaxis.set_major_formatter(PercentFormatter()); ax.yaxis.set_major_formatter(PercentFormatter())
ax.set_xlabel("share of item-store series (largest weight first)"); ax.set_ylabel("cumulative share of weight")
ax.set_title("Level-12 WRMSSE weight is concentrated")
ax.legend(loc="lower right")
fig.tight_layout(); save(fig, "q2_volume_and_weight"); plt.show()

share = by_store.sum() / by_store.sum().sum()
print("Unit share by category:", ", ".join(f"{c} {share[c]:.0%}" for c in cats))
state_share = units.groupby(meta.state_id).units.sum() / units.units.sum()
print("Unit share by state:   ", ", ".join(f"{s} {v:.0%}" for s, v in state_share.items()))
for k in [0.01, 0.10, 0.20]:
    print(f"Top {k:>4.0%} of item-stores carry {cum_w[int(k * n_series) - 1]:.0%} of the level-12 weight")
print(f"Item-stores with zero weight (no sales in the last 28 days): {(dollars == 0).sum():,}")

**How the hierarchy affects my modelling choices**

1. **The signal changes with the level.** Store, state and total series are high-volume, smooth and strongly seasonal. Item-store series are low-count and mostly zeros (quantified in Q3). One model family cannot be best everywhere. Classical models (ETS/ARIMA) fit the 154 smooth series of levels 1–9 well. The 30,490 intermittent bottom series need a count-aware, pooled model. Fitting 30k separate ARIMAs would be slow and inaccurate.
2. **The metric spreads weight over the levels.** Each level counts 1/12 of WRMSSE. Levels 1–9 (154 series) therefore carry 75% of the score, and the 30,490 item-stores only 1/12. Small biases that are invisible per item add up and are punished at the total, state and store levels, so forecasts must be unbiased in aggregate. Within a level, weights are recent dollar sales and are concentrated (chart above). High-revenue items matter most, and series with no recent sales carry no weight.
3. **Coherence.** Store, department and item plans must add up. Bottom-up from one global model is coherent by construction but carries bottom-level noise upwards. Reconciliation (WLS/MinT) or middle-out combines good aggregate forecasts with item-level detail.
4. **Grouped, not nested.** Because product × location is crossed, there is no single "middle" level. **Store × department (level 9, 70 series)** is the finest level that is still smooth and is a natural point for middle-out. Stores and departments are also natural partitions for per-store models.
5. **Pooling and hierarchy features.** One global model across items and stores learns shared seasonality and price/event responses, with item, department, category, store and state ids as categorical features. Rolling demand of each series' parents (the item across all stores, the store × department) lets a sparse series borrow information from its smoother ancestors.
6. **Per-level evaluation.** I report WRMSSE per level, not just the total, to see where a model wins or loses.

<a id="q3"></a>
## Q3. Exploratory analysis: trend, seasonality, calendar effects and intermittency

I look at representative series from the top of the hierarchy to the bottom. The rest of this section looks at each pattern in turn.

In [ ]:
upper_rows = np.where(info["level"] <= 9)[0]
U = pd.DataFrame(np.asarray(S[upper_rows] @ Y),
                 index=pd.MultiIndex.from_frame(info.loc[upper_rows, ["level", "series"]]))

def node(level, name):
    return pd.Series(U.loc[(level, name)].to_numpy(), index=dates)

total = node(1, "Total")
top_store = U.loc[3].sum(axis=1).idxmax()
l9 = U.loc[9]
top_store_dept = l9[l9.index.str.startswith(top_store + "_")].sum(axis=1).idxmax()

recent = Y[:, year]
on_sale_year = ~np.isnan(P_day[:, year]).any(axis=1)      # on sale for the whole last year
zero_share = (recent == 0).mean(axis=1)
in_store = np.where((meta.store_id == top_store).to_numpy() & on_sale_year)[0]
top_item = in_store[recent[in_store].sum(axis=1).argmax()]
typical_item = in_store[np.abs(zero_share[in_store] - np.median(zero_share[on_sale_year])).argmin()]

panels = [
    ("Level 1 · Total", total),
    (f"Level 3 · Store {top_store}", node(3, top_store)),
    (f"Level 9 · Store x dept {top_store_dept}", node(9, top_store_dept)),
    (f"Level 12 · {short_id(meta.id[top_item])} (top seller in {top_store})", pd.Series(Y[top_item], index=dates)),
    (f"Level 12 · {short_id(meta.id[typical_item])} (typical item-store)", pd.Series(Y[typical_item], index=dates)),
]
fig, axes = plt.subplots(len(panels), 1, figsize=(13, 12), sharex=True)
for ax, (title, s) in zip(axes, panels):
    ax.plot(s.index, s.to_numpy(), color=BLUE_LIGHT, lw=0.6, label="Daily units")
    ax.plot(s.index, s.where(~closed).rolling(28, min_periods=20).mean(), color=SERIES[0], lw=1.8,
            label="28-day mean")
    ax.set_title(title); thousands(ax)
axes[0].legend(loc="upper left", ncol=2)
fig.tight_layout(); save(fig, "q3_series_by_level"); plt.show()

**Representative series.** Going down the hierarchy:

- **Trend.** The total and the largest store grow over the five years. The store × department series follows its store. The top-selling item-store has its own life cycle, with level shifts that do not match the store's trend.
- **Noise.** Day-to-day variation is small relative to the level at the top. At the item-store level it dominates.
- **Christmas.** Every 25 December, sales at all levels drop to almost zero because stores are closed. Statistical models treat this as an outlier (imputed before fitting). LightGBM sees it as a known calendar event.
- **Intermittency.** The typical item-store sells nothing on many days, shown in more detail below.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.2))
last = slice(n_days - 112, n_days)
for ax, i, label in zip(axes, [top_item, typical_item], ["top seller", "typical item-store"]):
    ax.bar(dates[last], Y[i, last], color=SERIES[0], width=0.8)
    ax.set_title(f"{short_id(meta.id[i])} ({label}) · last 16 weeks")
    ax.set_ylabel("units")
fig.autofmt_xdate(); fig.tight_layout(); save(fig, "q3_bottom_level_zoom"); plt.show()
for i, label in [(top_item, "top seller"), (typical_item, "typical item-store")]:
    print(f"{label:>19}: zero-sales days in last year = {zero_share[i]:.0%}, mean units/day = {recent[i].mean():.2f}")
print(f"Median share of zero-sales days across item-stores on sale all year: {np.median(zero_share[on_sale_year]):.0%}")

### Trend: by state, category and store

In [ ]:
def monthly(s):
    s = s.where(~closed)
    return s.groupby(s.index.to_period("M")).mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
growth = {}
for ax, level, title in [(axes[0], 2, "Mean daily units by state (monthly)"),
                         (axes[1], 4, "Mean daily units by category (monthly)")]:
    names = U.loc[level].sum(axis=1).sort_values(ascending=False).index
    for i, name in enumerate(names):
        m = monthly(node(level, name))
        x = m.index.to_timestamp()
        ax.plot(x, m.to_numpy(), color=SERIES[i], label=name)
        yearly = node(level, name).where(~closed).groupby(dates.year).mean()
        growth[name] = yearly.get(2015) / yearly.get(2012) - 1
    ax.set_title(title); thousands(ax); ax.legend(loc="upper left", ncol=3)
fig.tight_layout(); save(fig, "q3_trend_state_category"); plt.show()
print("Growth in mean daily units, 2012 -> 2015:", ", ".join(f"{k} {v:+.0%}" for k, v in growth.items()))

In [ ]:
stores = U.loc[3].index.tolist()
fig, axes = plt.subplots(2, 5, figsize=(14, 5), sharex=True, sharey=True)
for ax, st in zip(axes.flat, stores):
    s = node(3, st).where(~closed).rolling(28, min_periods=20).mean()
    ax.plot(s.index, s.to_numpy(), color=SERIES[0], lw=1.4)
    ax.set_title(st, fontsize=10)
    thousands(ax)
fig.suptitle("28-day mean daily units by store", x=0.01, ha="left", fontweight="bold", fontsize=11)
fig.autofmt_xdate(); fig.tight_layout(); save(fig, "q3_trend_by_store"); plt.show()

**Trend.** CA is the largest state and FOODS the largest category by far (see the shares printed in Q2). Volume grows over the period, but at **different rates by state and category**. Stores differ even more: some grow steadily, others are flat, and several show **abrupt level shifts** (structural breaks such as remodels or new competitors; CA_2 is the usual example).

Consequences:

- Trend is local, not global. Statistical models are fitted on a recent window (2 years) so the level adapts to recent breaks.
- Tree models cannot extrapolate a trend. LightGBM therefore gets the *recent level* through rolling means of lagged sales and learns ratios relative to it. Store and department ids let it learn each store's dynamics. Per-store models are a natural extension.

### Seasonality: weekly, annual and within the month

In [ ]:
recent3 = np.arange(n_days) >= n_days - 3 * 364      # last three years
fig, axes = plt.subplots(1, 3, figsize=(14, 3.9))
keys = {"dow": cal_n["date"].dt.dayofweek, "month": cal_n["month"], "mday": cal_n["date"].dt.day}
for i, cat in enumerate(cats):
    s = node(4, cat).where(~closed)
    ratio = (s / s.rolling(364, center=True, min_periods=182).mean()).to_numpy()   # de-trended
    df = pd.DataFrame({"r": ratio, **{k: v.to_numpy() for k, v in keys.items()}})[recent3]
    for ax, k in zip(axes, keys):
        idx = df.groupby(k)["r"].mean()
        ax.plot(idx.index, idx.to_numpy(), marker="o", ms=4, color=SERIES[i], label=cat)
for ax, title in zip(axes, ["Day of week", "Month of year", "Day of month"]):
    ax.axhline(1.0, color=AXIS, lw=1)
    ax.set_title(f"{title} index (1 = trend level)")
axes[0].set_xticks(range(7), ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
axes[1].set_xticks(range(1, 13), list("JFMAMJJASOND"))
axes[2].set_xticks([1, 5, 10, 15, 20, 25, 31])
axes[0].legend(loc="upper left")
fig.tight_layout(); save(fig, "q3_seasonality_profiles"); plt.show()

In [ ]:
from statsmodels.tsa.seasonal import STL, MSTL

def strengths(y):
    # Strength of trend and weekly seasonality (Wang, Smith & Hyndman), from a robust STL fit
    r = STL(y, period=7, robust=True).fit()
    resid_var = np.var(r.resid)
    ft = max(0.0, 1 - resid_var / np.var(r.trend + r.resid))
    fs = max(0.0, 1 - resid_var / np.var(r.seasonal + r.resid))
    return ft, fs

rng = np.random.default_rng(0)
win2 = slice(n_days - 728, n_days)
cols = np.where(~np.isnan(P_day[:, win2]).any(axis=1))[0]  # series on sale for the whole window
S_cols, Y_cols = S[:, cols], Y[cols][:, win2]
rows = []
for level, (name, _) in core.LEVELS.items():
    idx = np.where(info["level"].to_numpy() == level)[0]
    pick = rng.choice(idx, size=min(40, len(idx)), replace=False)
    A = np.asarray(S_cols[pick] @ Y_cols)
    A = A[A.sum(axis=1) > 0]
    A = core.fill_closed_days(A, closed[win2])
    st = np.array([strengths(a) for a in A])
    rows.append({"level": level, "name": name, "series sampled": len(A),
                 "trend strength": np.median(st[:, 0]), "weekly seasonality strength": np.median(st[:, 1]),
                 "share of zero days": np.median((A == 0).mean(axis=1))})
signal = pd.DataFrame(rows)
display(signal)

fig, ax = plt.subplots(figsize=(12, 3.8))
for i, col in enumerate(["trend strength", "weekly seasonality strength", "share of zero days"]):
    ax.plot(signal["level"], signal[col], marker="o", ms=5, color=SERIES[i], label=col)
ax.set_xticks(signal["level"], [f"L{l}\n{core.LEVELS[l][0]}" for l in signal["level"]], fontsize=7)
ax.set_ylim(0, 1.05)
ax.set_title("How the signal changes down the hierarchy (median over sampled series, last 2 years)")
ax.legend(loc="upper right")
fig.tight_layout(); save(fig, "q3_signal_by_level"); plt.show()

In [ ]:
y_total = core.fill_closed_days(total.to_numpy()[None, :], closed)[0]
mstl = MSTL(pd.Series(y_total), periods=(7, 365)).fit()
seas = mstl.seasonal

fig, axes = plt.subplots(4, 1, figsize=(13, 10))
axes[0].plot(dates, y_total, color=BLUE_LIGHT, lw=0.6, label="Total (Christmas imputed)")
axes[0].plot(dates, mstl.trend, color=SERIES[0], lw=1.8, label="Trend")
axes[0].set_title("Level 1 · MSTL decomposition: data and trend"); axes[0].legend(loc="upper left", ncol=2)
axes[1].plot(dates, seas["seasonal_365"], color=SERIES[0], lw=1.2)
axes[1].set_title("Annual seasonal component")
wk = slice(n_days - 56, n_days)
axes[2].plot(dates[wk], seas["seasonal_7"].to_numpy()[wk], color=SERIES[0], marker="o", ms=3)
axes[2].set_title("Weekly seasonal component (last 8 weeks)")
axes[3].plot(dates, mstl.resid, color=SERIES[0], lw=0.6)
axes[3].set_title("Remainder: spikes line up with holidays and events")
top_resid = np.argsort(-np.abs(mstl.resid.to_numpy()))[:6]
for k in top_resid:
    label = cal_n.loc[k, "event_name_1"] if isinstance(cal_n.loc[k, "event_name_1"], str) else dates[k].strftime("%Y-%m-%d")
    axes[3].annotate(label, (dates[k], mstl.resid.iloc[k]), xytext=(3, 3), textcoords="offset points",
                     fontsize=7, color=INK2)
for ax in axes:
    thousands(ax)
fig.tight_layout(); save(fig, "q3_mstl_total"); plt.show()

shares = {c: np.var(seas[c]) for c in seas.columns}
shares["remainder"] = np.var(mstl.resid)
tot_var = sum(shares.values())
print("Share of de-trended variance at level 1:", ", ".join(f"{k} {v / tot_var:.0%}" for k, v in shares.items()))

**Seasonality.**

- **Weekly.** The weekly cycle is the dominant seasonal pattern at every level. Sales peak at the weekend and are lowest mid-week. The shape is similar across categories; the amplitude differs.
- **Annual.** The annual cycle is weaker. The month-of-year index moves less than the day-of-week index, and the MSTL decomposition shows a smaller annual component than weekly component.
- **Within the month.** The day-of-month profile is higher in the first half of the month for FOODS. That matches SNAP benefit timing (and paydays), so SNAP needs to be a feature.
- **Down the hierarchy.** The signal chart and table show weekly-seasonality strength and trend strength falling from the top to the bottom, while the share of zero days rises. At item-store level most of the variance is noise. Seasonality and trend cannot be estimated reliably series by series and must be learned from pooled data (global model) or estimated at aggregate levels and pushed down.

**Transformations this suggests.** The weekly pattern scales with the level (multiplicative), so at the top levels I model `log1p(sales)` in the ARIMA model. ETS uses an additive season on the original scale and is fitted only on a recent window. Christmas closures are imputed before fitting statistical models.

### Calendar and holiday effects: events and SNAP

In [ ]:
tot = total.to_numpy()
is_event = cal_n["event_name_1"].notna().to_numpy()
rows = []
for d in np.where(is_event)[0]:
    nb = [j for j in (d - 14, d - 7, d + 7, d + 14) if 0 <= j < n_days and not is_event[j] and not closed[j]]
    if nb:
        rows.append({"event": cal_n.loc[d, "event_name_1"], "type": cal_n.loc[d, "event_type_1"],
                     "lift": tot[d] / tot[nb].mean() - 1})
event_lift = pd.DataFrame(rows).groupby(["event", "type"])["lift"].agg(["mean", "count"]).sort_values("mean")

snap_rows = []
for st in ["CA", "TX", "WI"]:
    for cat in cats:
        s = node(6, f"{st}_{cat}").to_numpy()
        df = pd.DataFrame({"y": s, "snap": cal_n[f"snap_{st}"].to_numpy(),
                           "ym": cal_n["date"].dt.to_period("M").to_numpy(),
                           "dow": cal_n["date"].dt.dayofweek.to_numpy()})[recent3 & ~closed]
        g = df.groupby(["ym", "dow", "snap"])["y"].mean().unstack("snap").dropna()
        snap_rows.append({"state": st, "category": cat, "lift": (g[1] / g[0]).mean() - 1})
snap_lift = pd.DataFrame(snap_rows).pivot(index="state", columns="category", values="lift")[cats]

fig, axes = plt.subplots(1, 2, figsize=(14, 6.5), gridspec_kw={"width_ratios": [1.3, 1]})
ax = axes[0]
labels = [f"{e} ({t})" for e, t in event_lift.index]
vals = event_lift["mean"].to_numpy()
ax.barh(labels, vals, color=np.where(vals >= 0, POS, NEG), height=0.7)
ax.axvline(0, color=AXIS, lw=1)
ax.xaxis.set_major_formatter(PercentFormatter(1.0)); ax.grid(axis="x"); ax.grid(axis="y", visible=False)
ax.tick_params(axis="y", labelsize=7)
ax.set_title("Event-day total sales vs. same weekday ±1–2 weeks")
ax = axes[1]
xs = np.arange(len(snap_lift))
bw = 0.26
for i, cat in enumerate(cats):
    ax.bar(xs + (i - 1) * bw, snap_lift[cat], width=bw, color=SERIES[i], edgecolor=SURFACE, linewidth=1.5, label=cat)
ax.set_xticks(xs, snap_lift.index)
ax.axhline(0, color=AXIS, lw=1)
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title("SNAP-day lift by state and category\n(same month and weekday, last 3 years)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.06), ncol=3)
fig.tight_layout(); save(fig, "q3_calendar_effects"); plt.show()

print("Largest negative / positive event effects on total sales:")
display(pd.concat([event_lift.head(5), event_lift.tail(5)]).style.format({"mean": "{:+.1%}"}))
print("SNAP lift (state x category):")
display(snap_lift.style.format("{:+.1%}"))

**Calendar and holiday effects.**

- **Events.** Most events move total sales by a few percent, small next to the weekly cycle. A handful of national holidays move demand a lot: Christmas (closed, about −100%) and typically Thanksgiving and New Year. The remainder spikes in the MSTL chart line up with these days. Effects differ by event and by category, and some events also shift demand to the days *before* them.
- **SNAP.** SNAP days raise **FOODS** sales in every state and have little effect on HOBBIES and HOUSEHOLD. Each state has its own SNAP calendar, so the effect is **state-specific** and interacts with category.

**Modelling consequences.** Event name and event type go in as categorical features for LightGBM, so each event can get its own effect per item through the trees. ARIMA gets event-type and Christmas dummies. SNAP enters as the flag *of the series' own state*, and the trees learn its interaction with the category id. Calendar features are known in advance for the forecast period, so they are legitimate inputs at any horizon.

### Intermittency

In [ ]:
Yw = Y[on_sale_year][:, year]
nz = (Yw > 0).sum(axis=1)
ok = nz >= 2
adi = Yw.shape[1] / nz[ok]                                 # average inter-demand interval
size_mean = Yw[ok].sum(axis=1) / nz[ok]
size_var = (Yw[ok] ** 2).sum(axis=1) / nz[ok] - size_mean ** 2
cv2 = np.clip(size_var, 0, None) / size_mean ** 2           # squared CV of non-zero demand sizes
classes = pd.Series(np.select([(adi < 1.32) & (cv2 < 0.49), (adi < 1.32), (cv2 < 0.49)],
                              ["smooth", "erratic", "intermittent"], "lumpy"))
class_share = classes.value_counts(normalize=True).reindex(["smooth", "erratic", "intermittent", "lumpy"]).fillna(0)

zero_by_level = []
S_on = S[:, on_sale_year]
Y_on = Y[on_sale_year][:, year]
keep_days = ~closed[year]
for level, (name, _) in core.LEVELS.items():
    A = np.asarray(S_on[np.where(info["level"].to_numpy() == level)[0]] @ Y_on)[:, keep_days]
    zero_by_level.append((level, (A == 0).mean()))
zero_by_level = pd.DataFrame(zero_by_level, columns=["level", "share of zero days"])

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), gridspec_kw={"width_ratios": [1, 1.1, 1]})
ax = axes[0]
ax.hist((Yw == 0).mean(axis=1), bins=np.linspace(0, 1, 21), color=SERIES[0], edgecolor=SURFACE, linewidth=1.5)
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title("Share of zero-sales days per item-store"); ax.set_xlabel("zero days in last 52 weeks"); ax.set_ylabel("series")
ax = axes[1]
ax.scatter(adi, np.clip(cv2, 0.01, None), s=3, alpha=0.15, color=SERIES[0], linewidths=0)
ax.set_xscale("log"); ax.set_yscale("log")
ax.axvline(1.32, color=AXIS, lw=1); ax.axhline(0.49, color=AXIS, lw=1)
for label, (x, y) in {"smooth": (0.03, 0.04), "erratic": (0.03, 0.93), "intermittent": (0.6, 0.04),
                      "lumpy": (0.6, 0.93)}.items():
    ax.text(x, y, f"{label} {class_share[label]:.0%}", transform=ax.transAxes, fontsize=9, color=INK2,
            fontweight="bold", va="top" if y > 0.5 else "bottom")
ax.set_xlabel("ADI (average days between sales)"); ax.set_ylabel("CV² of non-zero sales (floored at 0.01)")
ax.set_title("Syntetos–Boylan demand classes"); ax.grid(axis="x")
ax = axes[2]
ax.bar(zero_by_level["level"], zero_by_level["share of zero days"], color=SERIES[0], width=0.7)
for lvl, v in zero_by_level.itertuples(index=False):
    if v >= 0.005:
        ax.annotate(f"{v:.0%}", (lvl, v), xytext=(0, 3), textcoords="offset points", ha="center", fontsize=8, color=INK2)
ax.set_xticks(zero_by_level["level"], [f"L{l}" for l in zero_by_level["level"]])
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title("Zero days by hierarchy level")
fig.tight_layout(); save(fig, "q3_intermittency"); plt.show()

print("Demand classes of item-stores (last 52 weeks):", ", ".join(f"{k} {v:.0%}" for k, v in class_share.items()))
print(f"Item-stores with fewer than 2 sale days in the last year (not classified): {(~ok).sum():,}")
print(f"Share of all item-store-days with zero sales (whole history, incl. not-yet-launched): {(Y == 0).mean():.0%}")

**Intermittency.**

- **Bottom level.** Most item-stores have zero sales on a large share of days. In the Syntetos–Boylan classification (ADI = 1.32 and CV² = 0.49 cut-offs) the majority are *intermittent* or *lumpy*, and few are *smooth*.
- **Aggregation removes it.** Zero days vanish by the item level (L10) and are absent at store level. Over the whole file the share of zeros is inflated further by days before an item's launch.

**Modelling consequences.**

1. **Loss.** Squared error on raw counts, or Gaussian ARIMA, fits zero-inflated, skewed demand poorly. I use a **Tweedie** objective (compound Poisson–gamma: a mass at zero plus a skewed positive part). It predicts the *mean*, which is what a squared-error metric rewards. Croston/TSB are reasonable per-series baselines.
2. **Data.** Zeros while an item is on sale are real demand and stay in the training data. Days before launch (no price) are dropped.
3. **Level-specific models.** Smooth aggregate levels are where ETS/ARIMA shine, so they supply the upper-level forecasts for reconciliation.
4. **Metric.** RMSSE scales errors by each series' own naive error, so intermittent series are judged on a comparable scale.

<a id="q4"></a>
## Q4. Models, transformations and exogenous variables

**Model specification**

| Model | Where in the hierarchy | Role | Transformation | Exogenous inputs |
|---|---|---|---|---|
| Seasonal naive (repeat last week) | every series | benchmark | — | — |
| **ETS(A, Ad, A)**: damped additive trend, additive weekly season (Holt–Winters) | levels 1–9 (154 smooth series) | upper-level base forecasts | Christmas closures imputed; fitted on the last 2 years so the level adapts to breaks | none (univariate) |
| **Dynamic-regression ARIMA (SARIMAX)** | level 1 here; any smooth aggregate | shows exogenous effects explicitly and interpretably | `log1p` (multiplicative weekly pattern, variance stabilisation); SARIMA(1,1,1)(0,1,1)₇, i.e. regular and weekly differencing for trend and weekly season | SNAP flags (CA/TX/WI), event-type dummies, Christmas dummy, annual Fourier terms |
| **Global LightGBM, Tweedie loss** | all 30,490 item-stores (level 12) | main model | no target transform (Tweedie log link handles skew and zeros); prices as ratios | price level, price relative to item's max and mean price (promotion depth), week-over-week price change, events, state SNAP, calendar, hierarchy demand features |
| **WLS reconciliation + middle-out** | levels 1–9 → 12 | coherent combination of statistical and ML forecasts | — | — |

**Why each choice**

- **Global LightGBM** fits the data shape. There are 30k short, noisy, intermittent series that share patterns: weekly cycle, SNAP, events, price response. One model trained on all of them learns these once and applies them everywhere. That includes new or sparse items, which a per-series model cannot handle. Trees capture non-linear interactions (SNAP × FOODS × state, event × category, discount depth × item) without hand-crafting. LightGBM also scales to tens of millions of rows.
- **Tweedie loss.** Demand is a non-negative count with many zeros and a long right tail. Tweedie (variance power 1.1, between Poisson and gamma) models exactly that and returns the conditional mean.
- **Direct strategy with lags ≥ 28 days.** All sales-based features (lags 28–56, rolling means and std of the series, item and store-department rolling means) end at least 28 days before the target day. One model then serves all 28 horizons without feeding its own predictions back (no error accumulation), and validation uses the same information as the real forecast. The cost is not using the most recent 27 days. Horizon-specific models (lags ≥ h) or a recursive model in an ensemble can recover some of that.
- **Exogenous variables (all known in advance in M5).**
  - **Price and promotions:** current price, price ÷ the item-store's maximum and mean price (a discount signal, since M5 has no promotion flag), week-over-week price change and number of distinct prices.
  - **Events:** name and type, as categoricals.
  - **SNAP:** the flag for the series' own state.
  - **Calendar:** weekday, month, day of month, week of year.
  - **Life cycle:** weeks since the item launched in the store.
  - **Hierarchy:** item ids and parent-level demand.
- **ETS/ARIMA at upper levels.** These series are smooth, strongly seasonal and long, which is where statistical models are strong, fast and interpretable. ETS adapts its level to breaks. SARIMAX quantifies the SNAP and event effects, which justifies those features.
- **Reconciliation.** Upper-level forecasts (ETS, or a combination of ETS and aggregated LightGBM) are reconciled with **WLS using structural scaling**, W = diag(number of item-stores under each node), over levels 1–9. The coherent store × department forecasts are then split to items in proportion to the LightGBM forecasts (**middle-out**). Every level is coherent, and the holdout decides between this and pure bottom-up.
- **Considered alternatives.**
  - Croston/TSB/ADIDA: good intermittent baselines, but they ignore prices and events.
  - Per-series ARIMA at the bottom: too slow and too noisy for 30k series.
  - Deep global models (DeepAR, N-BEATS, Temporal Fusion Transformer): competitive but costly to tune.
  - Vertex AI **AutoML forecasting** with hierarchical options: a managed benchmark, but less control over features, loss and validation.

**What changed compared with my first LightGBM notebook (`lgbm-model.ipynb`)**

| Issue in the first version | Effect | Fix here |
|---|---|---|
| `lag_7` and rolling stats with `shift(1)` were computed on actual sales *inside* the 28-day validation window | Validation used information unavailable at forecast time, so its scores were too optimistic. At submission the same features were filled recursively with predictions, a different regime from training. | All sales features lag ≥ 28 days (direct strategy). A leakage test confirms features don't change when the target window changes. |
| Optuna picked the `quantile` objective with α ≈ 0.2 | Forecasts the 20th percentile, a systematic under-forecast for a squared-error metric | Tweedie objective (conditional mean) |
| Optuna ran **one** trial: the first fit on 58M rows took ~4 h, longer than the 2 h timeout | "Best" parameters were the first random draw | Fixed sensible parameters and early stopping on a separate fold. Tune later on a subsample or one store. |
| Early stopping and scoring on the same 28 days | Optimistic holdout score | Fold A for early stopping, fold B untouched for scoring |
| Events and prices set to 0 for the forecast days | Train/serve skew, although calendar and prices for d_1942–1969 are in the data | Use the known future calendar and prices |
| Rows sorted by `id` alphabetically before scoring with `m5_wrmsse` | The scorer expects `sales_train_evaluation.csv` order, so series were likely mismatched | WRMSSE computed in-notebook on rows kept in file order |
| Training rows included days before item launch | Structural zeros mixed with real zero demand | Rows without a sell price are dropped |

### 4.1 Evidence for the price / promotion features

In [ ]:
with np.errstate(invalid="ignore", divide="ignore"):
    p = P_day[:, year]
    rel_price = p / np.nanmax(p, axis=1, keepdims=True)               # 1.0 = regular (highest) price
    y_rel = recent / np.nanmean(np.where(np.isnan(p), np.nan, recent), axis=1, keepdims=True)
valid = ~np.isnan(rel_price) & np.isfinite(y_rel)
bins = [0, 0.7, 0.8, 0.9, 0.95, 0.99, 1.01]
labels = ["<70%", "70–80%", "80–90%", "90–95%", "95–99%", "regular"]
binned = pd.DataFrame({"bin": pd.cut(rel_price[valid], bins, labels=labels), "y": y_rel[valid]})
price_effect = binned.groupby("bin", observed=False)["y"].agg(["mean", "size"])
price_effect["share of item-store-days"] = price_effect["size"] / price_effect["size"].sum()

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.bar(price_effect.index.astype(str), price_effect["mean"], color=SERIES[0], width=0.6)
ax.axhline(1.0, color=AXIS, lw=1)
for i, v in enumerate(price_effect["mean"]):
    if np.isfinite(v):
        ax.annotate(f"{v:.2f}", (i, v), xytext=(0, 3), textcoords="offset points", ha="center", fontsize=8, color=INK2)
ax.set_xlabel("price as % of the item-store's highest price in the last year")
ax.set_ylabel("demand ÷ item-store average")
ax.set_title("Lower relative prices come with higher demand")
fig.tight_layout(); save(fig, "q4_price_effect"); plt.show()
display(price_effect.style.format({"mean": "{:.2f}", "size": "{:,}", "share of item-store-days": "{:.1%}"}))

Demand relative to an item's own average rises as its price falls below its regular level. Promotions are not flagged in M5, but price ratios capture them. That is why the model gets **relative** price features (`price_rel_max`, `price_rel_mean`, `price_momentum`) and not just the raw price, which mostly identifies the item.

### 4.2 Backtest design and features

- **Fold A** (early stopping): train on the `LOOKBACK_DAYS` days ending d_1885, validate on d_1886–1913. This picks the number of boosting rounds.
- **Fold B** (holdout, never used for any choice before scoring): retrain on the window ending d_1913 with that number of rounds and forecast d_1914–1941. All strategies are scored with WRMSSE here.
- **Final:** retrain on the window ending d_1941 and forecast d_1942–1969 (the Kaggle evaluation period) with the strategy that won fold B.

In [ ]:
t0 = time.time()
fb = core.FeatureBuilder(meta, Y, calendar, P_week, day_week)
print(f"Feature builder ready in {time.time() - t0:.0f}s; {len(fb.feature_names)} features:")
print(", ".join(fb.feature_names))
print("categorical:", ", ".join(fb.categorical_features))

HOLDOUT_END = n_days - core.HORIZON        # d_1913: holdout is d_1914..d_1941
TUNE_END = HOLDOUT_END - core.HORIZON      # d_1885: early-stopping fold is d_1886..d_1913

def days(first, last):
    return np.arange(first, last + 1)

# Leakage guard: sales-based features for the holdout must not depend on holdout sales
X_a, _, _ = fb.make(days(HOLDOUT_END + 1, n_days)[:3], require_target=False)
Y_perturbed = Y.copy(); Y_perturbed[:, HOLDOUT_END:] += 5
fb_check = core.FeatureBuilder(meta, Y_perturbed, calendar, P_week, day_week)
X_b, _, _ = fb_check.make(days(HOLDOUT_END + 1, n_days)[:3], require_target=False)
assert np.array_equal(np.nan_to_num(X_a, nan=-1), np.nan_to_num(X_b, nan=-1))
print("Leakage check passed: holdout features are identical when holdout sales change.")
del X_a, X_b, Y_perturbed, fb_check
_ = gc.collect()

### 4.3 Fold A: early stopping

In [ ]:
t0 = time.time()
model_a = core.train_lgbm(fb, days(TUNE_END - LOOKBACK_DAYS + 1, TUNE_END), days(TUNE_END + 1, HOLDOUT_END),
                          params=LGB_OVERRIDES, num_boost_round=MAX_ROUNDS)
best_rounds = model_a.best_iteration or MAX_ROUNDS
print(f"Best number of rounds: {best_rounds}  ({(time.time() - t0) / 60:.1f} min)")

### 4.4 Fold B: holdout scores for every strategy

In [ ]:
t0 = time.time()
model_b = core.train_lgbm(fb, days(HOLDOUT_END - LOOKBACK_DAYS + 1, HOLDOUT_END),
                          params=LGB_OVERRIDES, num_boost_round=best_rounds)
holdout_days = days(HOLDOUT_END + 1, n_days)
lgb_holdout = core.predict_days(model_b, fb, holdout_days)

evaluator = core.WRMSSE(Y, P_day, S, info, train_end=HOLDOUT_END)
variants, ets_up = core.hierarchical_variants(lgb_holdout, Y, S, info, HOLDOUT_END, closed)
candidates = {"Seasonal naive": core.seasonal_naive(Y, HOLDOUT_END), **variants}

scores = {name: evaluator.score(f) for name, f in candidates.items()}
by_level = pd.DataFrame({name: s[1] for name, s in scores.items()})
by_level.index = [f"L{l} {core.LEVELS[l][0]}" for l in by_level.index]
by_level.loc["WRMSSE (mean of levels)"] = pd.Series({name: s[0] for name, s in scores.items()})
best_name = min(variants, key=lambda k: scores[k][0])
print(f"Holdout d_{HOLDOUT_END + 1}..d_{n_days} ({(time.time() - t0) / 60:.1f} min). Best strategy: {best_name}")
display(by_level.style.format("{:.3f}").highlight_min(axis=1, props="font-weight:bold"))

In [ ]:
levels = list(core.LEVELS)
fig, ax = plt.subplots(figsize=(13, 4.2))
for i, name in enumerate(candidates):
    ax.plot(levels, [scores[name][1][l] for l in levels], marker="o", ms=5, color=SERIES[i],
            label=f"{name} (WRMSSE {scores[name][0]:.3f})")
ax.set_xticks(levels, [f"L{l}\n{core.LEVELS[l][0]}" for l in levels], fontsize=7)
ax.set_ylabel("level WRMSSE (lower is better)")
ax.set_title("Holdout accuracy by hierarchy level")
ax.legend(loc="upper left")
fig.tight_layout(); save(fig, "q4_wrmsse_by_level"); plt.show()

**Reading the table and chart.** Each row is one level's weighted RMSSE, and the overall score is their mean. The comparison shows *where* each approach wins:

- **LightGBM vs. seasonal naive** shows the value of the features (prices, events, SNAP, pooled seasonality), especially at the noisy bottom levels.
- **ETS reconciled + middle-out** typically does well at the smooth top levels. Its bottom levels inherit the LightGBM item mix, scaled to the reconciled store × department totals.
- **LightGBM + ETS** combines both model families before reconciliation. The strategy with the lowest overall holdout WRMSSE is used for the final forecast.

### 4.5 Dynamic-regression ARIMA at level 1: what the exogenous variables are worth

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

def level1_exog(cal):
    X = pd.DataFrame(index=cal.index)
    for st in ["CA", "TX", "WI"]:
        X[f"snap_{st}"] = cal[f"snap_{st}"].astype(float)
    for typ in ["Sporting", "Cultural", "National", "Religious"]:
        X[f"event_{typ}"] = ((cal["event_type_1"] == typ) | (cal["event_type_2"] == typ)).astype(float)
    X["christmas"] = ((cal["date"].dt.month == 12) & (cal["date"].dt.day == 25)).astype(float)
    doy = cal["date"].dt.dayofyear.to_numpy()
    for k in (1, 2, 3):
        X[f"sin_{k}"] = np.sin(2 * np.pi * k * doy / 365.25)
        X[f"cos_{k}"] = np.cos(2 * np.pi * k * doy / 365.25)
    return X

fit_start = HOLDOUT_END - 3 * 364
exog = level1_exog(calendar.iloc[:n_days].reset_index(drop=True))
t0 = time.time()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    sarimax = SARIMAX(np.log1p(tot[fit_start:HOLDOUT_END]), exog=exog.iloc[fit_start:HOLDOUT_END].to_numpy(),
                      order=(1, 1, 1), seasonal_order=(0, 1, 1, 7)).fit(disp=False, maxiter=300)
sarimax_fc = np.expm1(sarimax.forecast(core.HORIZON, exog=exog.iloc[HOLDOUT_END:n_days].to_numpy()))
print(f"SARIMAX(1,1,1)(0,1,1)[7] with {exog.shape[1]} regressors fitted in {time.time() - t0:.0f}s; "
      f"optimizer converged: {sarimax.mle_retvals.get('converged')}")

n_exog = exog.shape[1]   # exogenous coefficients come first in the parameter vector
effects = pd.DataFrame({"coef (log scale)": sarimax.params[:n_exog], "p-value": sarimax.pvalues[:n_exog]},
                       index=exog.columns)
effects["effect on sales"] = np.expm1(effects["coef (log scale)"])
display(effects.loc[[c for c in exog.columns if not c.startswith(("sin", "cos"))]]
        .style.format({"coef (log scale)": "{:+.4f}", "p-value": "{:.3f}", "effect on sales": "{:+.1%}"}))

actual_l1 = evaluator.actual[0]
l1 = {"SARIMAX (log1p, exog)": sarimax_fc, "ETS": ets_up[0],
      "LightGBM bottom-up": np.asarray(S[0] @ lgb_holdout).ravel()}
if best_name != "LightGBM bottom-up":
    l1[best_name] = np.asarray(S[0] @ variants[best_name]).ravel()
l1_rmsse = {k: np.sqrt(np.mean((actual_l1 - v) ** 2) / evaluator.scale[0]) for k, v in l1.items()}
display(pd.Series(l1_rmsse, name="Level-1 RMSSE (holdout)").to_frame().style.format("{:.3f}"))

hist = slice(HOLDOUT_END - 56, HOLDOUT_END)
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(dates[hist], tot[hist], color=INK, lw=1.2, label="Actual")
ax.plot(dates[HOLDOUT_END:n_days], actual_l1, color=INK, lw=1.2)
for i, (k, v) in enumerate(list(l1.items())[:3]):
    ax.plot(dates[HOLDOUT_END:n_days], v, color=SERIES[i], label=f"{k} (RMSSE {l1_rmsse[k]:.2f})")
ax.axvline(dates[HOLDOUT_END], color=AXIS, lw=1)
ax.set_title("Level 1 · holdout forecasts"); thousands(ax); ax.legend(loc="upper left", ncol=2)
fig.tight_layout(); save(fig, "q4_level1_holdout"); plt.show()

The SARIMAX coefficients are on the log scale, so `exp(coef) − 1` is the percentage effect on total sales. They put numbers on the Q3 findings: the SNAP and event effects, and the Christmas closure. Variables with a clear, significant effect at the aggregate are the ones worth giving the item-level model. The level-1 RMSSE table compares one interpretable statistical model with the ML model aggregated bottom-up.

### 4.6 What the LightGBM model uses

In [ ]:
imp = pd.Series(model_b.feature_importance("gain"), index=fb.feature_names)
imp = (imp / imp.sum()).sort_values(ascending=True).tail(20)
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(imp.index, imp.to_numpy(), color=SERIES[0], height=0.7)
ax.xaxis.set_major_formatter(PercentFormatter(1.0)); ax.grid(axis="x"); ax.grid(axis="y", visible=False)
ax.set_title("LightGBM feature importance (share of total gain, top 20)")
fig.tight_layout(); save(fig, "q4_feature_importance"); plt.show()

### 4.7 Holdout forecasts at four levels

In [ ]:
def agg_row(level, name):
    return int(info.index[(info["level"] == level) & (info["series"] == name)][0])

show = [(1, "Total"), (3, top_store), (9, top_store_dept), (12, short_id(meta.id[top_item]))]
best_fc = candidates[best_name]
naive_fc = candidates["Seasonal naive"]
fig, axes = plt.subplots(2, 2, figsize=(14, 7))
for ax, (level, name) in zip(axes.flat, show):
    r = agg_row(level, name)
    hist_vals = np.asarray(S[r] @ Y[:, HOLDOUT_END - 56:n_days]).ravel()
    ax.plot(dates[HOLDOUT_END - 56:n_days], hist_vals, color=INK, lw=1.1, label="Actual")
    ax.plot(dates[HOLDOUT_END:n_days], np.asarray(S[r] @ best_fc).ravel(), color=SERIES[0], label=best_name)
    ax.plot(dates[HOLDOUT_END:n_days], np.asarray(S[r] @ naive_fc).ravel(), color=SERIES[1], label="Seasonal naive")
    ax.axvline(dates[HOLDOUT_END], color=AXIS, lw=1)
    ax.set_title(f"L{level} · {name}"); thousands(ax)
axes[0, 0].legend(loc="upper left", fontsize=7)
fig.autofmt_xdate(); fig.tight_layout(); save(fig, "q4_holdout_forecasts"); plt.show()

### 4.8 Final model and submission

In [ ]:
t0 = time.time()
model_final = core.train_lgbm(fb, days(n_days - LOOKBACK_DAYS + 1, n_days),
                              params=LGB_OVERRIDES, num_boost_round=best_rounds)
future_days = days(n_days + 1, n_days + core.HORIZON)
lgb_future = core.predict_days(model_final, fb, future_days)
final_variants = core.hierarchical_variants(lgb_future, Y, S, info, n_days, closed)[0]
final_fc = final_variants[best_name]

submission = core.make_submission(meta, Y, final_fc)
submission.to_csv(os.path.join(OUTPUT_DIR, "submission.csv"), index=False)
model_final.save_model(os.path.join(OUTPUT_DIR, "lgbm_m5.txt"))
print(f"Strategy: {best_name}; rounds: {best_rounds}; {(time.time() - t0) / 60:.1f} min")
print(f"submission.csv: {submission.shape[0]:,} rows "
      f"(validation rows = actual d_{n_days - 27}..d_{n_days}, evaluation rows = forecast d_{n_days + 1}..d_{n_days + 28})")
print(f"Forecast total units over the 28 days: {final_fc.sum():,.0f} "
      f"(last 28 actual days: {Y[:, -28:].sum():,.0f})")
display(submission.head())

### 4.9 Score on the true evaluation period (when the actuals are available)

After the competition the organisers published the actual sales for d_1942–1969 (`sales_test_evaluation.csv`) and the official series weights (`weights_evaluation.csv`) in the [M5-methods repository](https://github.com/Mcompetitions/M5-methods). If they are in `DATA_PATH`, this cell:

- checks that the in-notebook WRMSSE weights match the official ones;
- scores every strategy on the real test period, i.e. the private-leaderboard metric.

Reference points from the official results (`Scores and Ranks.xlsx`): **winner 0.520**, 5th place 0.536, best statistical benchmark ES_bu 0.671, sNaive 0.847, Naive 1.752.

In [ ]:
test_path = os.path.join(DATA_PATH, "sales_test_evaluation.csv")
if not os.path.exists(test_path):
    print("sales_test_evaluation.csv not found in DATA_PATH; skipping the final scoring.")
else:
    test = pd.read_csv(test_path)
    if "id" not in test.columns:
        test["id"] = test["item_id"] + "_" + test["store_id"] + "_evaluation"
    test = test.set_index("id").loc[meta["id"]]                     # align rows with `meta`
    Y_full = np.concatenate([Y, test[[f"d_{d}" for d in future_days]].to_numpy(np.float32)], axis=1)
    final_eval = core.WRMSSE(Y_full, P_day, S, info, train_end=n_days)

    w_path = os.path.join(DATA_PATH, "weights_evaluation.csv")
    if os.path.exists(w_path):
        official = pd.read_csv(w_path)
        official["level"] = official["Level_id"].str.replace("Level", "").astype(int)
        gap = max(np.abs(np.sort(official.loc[official.level == l, "weight"].to_numpy())
                         - np.sort(final_eval.weights[info["level"].to_numpy() == l])).max() for l in core.LEVELS)
        print(f"Largest difference between notebook and official WRMSSE weights: {gap:.2e}")

    final_candidates = {"Seasonal naive": core.seasonal_naive(Y, n_days), **final_variants}
    final_scores = {name: final_eval.score(f) for name, f in final_candidates.items()}
    test_table = pd.DataFrame({name: s_[1] for name, s_ in final_scores.items()})
    test_table.index = [f"L{l} {core.LEVELS[l][0]}" for l in test_table.index]
    test_table.loc["WRMSSE (mean of levels)"] = pd.Series({name: s_[0] for name, s_ in final_scores.items()})
    print(f"Strategy chosen on the holdout: {best_name} -> test WRMSSE {final_scores[best_name][0]:.4f}")
    display(test_table.style.format("{:.3f}").highlight_min(axis=1, props="font-weight:bold"))

## Summary

1. **Approach (Q1).**
   - Reproduce the metric first.
   - Validate with rolling 28-day folds that mimic the real forecast.
   - Benchmark against seasonal naive and ETS.
   - Train one global LightGBM on all item-stores with horizon-safe features.
   - Make forecasts coherent with reconciliation, and choose the strategy by holdout WRMSSE per level.
   - Productionise as a Vertex AI custom training job.
2. **Hierarchy (Q2).** Product (3 categories → 7 departments → 3,049 items) is crossed with location (3 states → 10 stores), giving 12 levels and 42,840 series. Because each level counts equally in WRMSSE and the signal changes by level, I use statistical models where series are smooth (levels 1–9), a pooled count model at the bottom, and reconciliation to tie them together.
3. **EDA (Q3).**
   - Growing but store-specific trends, with breaks.
   - A dominant weekly cycle, a weaker annual one, and an early-month FOODS bump from SNAP.
   - Christmas closures, a few strong holidays and many small events.
   - Heavy intermittency at the bottom that disappears with aggregation.
4. **Models (Q4).**
   - Tweedie LightGBM with lags ≥ 28, rolling and hierarchy demand features, relative price/promotion features, events, state SNAP and calendar.
   - ETS and a log-scale SARIMAX with exogenous regressors for the aggregates.
   - WLS-structural reconciliation and middle-out.

**Running on the platform.** `m5_vertex_custom_training.ipynb` packages `m5_core.py` with a small `trainer/task.py` and runs this pipeline (holdout, refit, forecast) as a Vertex AI custom training job. It writes `submission.csv`, `model.txt` and `metrics.json` to Cloud Storage.